In [12]:
import pandas as pd
import os

# 1. Cargar los datos procesados
ruta_accesos = "../../datos/procesados/accesos_historico_total.csv"
ruta_socios = "../../datos/procesados/socios_historico_total.csv"

df_accesos = pd.read_csv(ruta_accesos)
df_socios = pd.read_csv(ruta_socios)

# 2. Preparación de fechas
df_accesos['fecha'] = pd.to_datetime(df_accesos['fecha'])
fecha_referencia = df_accesos['fecha'].max()

# --- FASE 1: INGENIERÍA DE CARACTERÍSTICAS GEOGRÁFICAS ---

def asignar_zona_proximidad(cp):
    try:
        cp_int = int(float(cp))
    except:
        return 2
    
    # Zona 0: Pineda de Mar (Local)
    if cp_int == 8397:
        return 0
    # Zona 1: Maresme (Cercano: Calella, Sta. Susanna, Malgrat)
    elif cp_int in [8370, 8398, 8380]:
        return 1
    # Zona 2: Otros / Larga distancia
    else:
        return 2

# Aplicamos la proximidad sobre la tabla de socios
df_socios['zona_proximidad'] = df_socios['codigo_postal'].apply(asignar_zona_proximidad)

# --- FASE 2: FEATURE ENGINEERING RFM ---

# Calculamos Recencia y Frecuencia
comportamiento = df_accesos.groupby('id_socio').agg(
    ultima_visita=('fecha', 'max'),
    total_visitas=('fecha', 'count')
).reset_index()

comportamiento['dias_desde_ultima_visita'] = (fecha_referencia - comportamiento['ultima_visita']).dt.days

# Día de la semana favorito (Estacionalidad)
df_accesos['dia_semana'] = df_accesos['fecha'].dt.dayofweek
dia_favorito = df_accesos.groupby('id_socio')['dia_semana'].apply(
    lambda x: x.mode()[0] if not x.mode().empty else -1
).reset_index()
dia_favorito.rename(columns={'dia_semana': 'dia_favorito'}, inplace=True)

# Unimos métricas de comportamiento
comportamiento = pd.merge(comportamiento, dia_favorito, on='id_socio', how='left')

# --- FASE 3: UNIFICACIÓN FINAL (ABT) ---

# Cruzamos socios (con su nueva zona) con comportamiento
df_abt = pd.merge(df_socios, comportamiento, on='id_socio', how='left')

# Rellenamos nulos con lógica de negocio
df_abt['total_visitas'] = df_abt['total_visitas'].fillna(0)
df_abt['dias_desde_ultima_visita'] = df_abt['dias_desde_ultima_visita'].fillna(999)
df_abt['dia_favorito'] = df_abt['dia_favorito'].fillna(-1)

# Eliminamos columnas temporales que no usaremos en el modelo
df_abt = df_abt.drop(columns=['ultima_visita'])

print("Lógica completada. Tabla ABT lista en memoria.")

Lógica completada. Tabla ABT lista en memoria.


In [14]:
# --- ANÁLISIS EXPLORATORIO RÁPIDO ---

print(f"Total de socios en la tabla: {len(df_abt)}\n")

# 1. Ver los 10 códigos postales que más socios aportan
print("--- TOP 10 CÓDIGOS POSTALES ---")
print(df_abt['codigo_postal'].value_counts().head(10))

# 2. Ver cuántos socios tenemos en cada una de las 3 zonas creadas
print("\n--- DISTRIBUCIÓN POR ZONAS ---")
print(df_abt['zona_proximidad'].value_counts().sort_index())

# 3. Ver zona con más % de baja (Churn)
print("\n--- % DE CHURN (BAJAS) POR ZONA ---")
print(df_abt.groupby('zona_proximidad')['es_baja'].mean() * 100)

Total de socios en la tabla: 8658

--- TOP 10 CÓDIGOS POSTALES ---
codigo_postal
08397     3307
8397.0    2088
08370      942
8370.0     823
08395      148
08398      141
8395.0     137
8398.0     115
08396      107
8396.0      89
Name: count, dtype: int64

--- DISTRIBUCIÓN POR ZONAS ---
zona_proximidad
0    5397
1    2169
2    1092
Name: count, dtype: int64

--- % DE CHURN (BAJAS) POR ZONA ---
zona_proximidad
0    38.688160
1    46.150300
2    48.626374
Name: es_baja, dtype: float64


In [15]:
# --- FASE FINAL: GUARDADO DE LA ABT ---

ruta_guardado = "../../datos/procesados"
os.makedirs(ruta_guardado, exist_ok=True)

ruta_final = f"{ruta_guardado}/abt_socios_modelo.csv"
df_abt.to_csv(ruta_final, index=False)

print(f"Archivo procesado y guardado físicamente con éxito en:\n{ruta_final}")

Archivo procesado y guardado físicamente con éxito en:
../../datos/procesados/abt_socios_modelo.csv
